# OpenACC

## Level 0: Hello World

We start with a serial CPU code printing a range of numbers:

```c++
for (size_t i0 = 0; i0 < nx; ++i0) {
    printf("%ld\n", i0);
}
```

The full example is available in [print-numbers-base.cpp](../src/print-numbers/print-numbers-base.cpp), and can be compiled and executed using the following cells.

In [ ]:
!g++ -O3 -march=native -std=c++17 -o ../build/print-numbers/print-numbers-base ../src/print-numbers/print-numbers-base.cpp

In [ ]:
!../build/print-numbers/print-numbers-base

OpenACC offers a similar approach to OpenMP target offloading, using `parallel` (to spawn threads) and `loop` (to distribute work).

Whether execution occurs on the CPU or GPU is determined by compiler arguments.

If not specified, the compiler chooses the number of gangs, workers, and vector size.

```c++
#pragma acc parallel loop
for (size_t i0 = 0; i0 < nx; ++i0) {
    printf("%ld\n", i0);
}   // implicit synchronization
```

As with OpenMP, this code instructs the compiler to parallelize the loop - regardless of whether it is safe (e.g., in the presence of race conditions due to inter-iteration dependencies).
Alternatively, `kernels` can be used to give the compiler more control.
In this case, the compiler will:
* Analyze dependencies and only parallelize loops without dependencies
* Apply loop and kernel transformations, including fusion

```c++
#pragma acc kernels
{
    for (size_t i0 = 0; i0 < nx; ++i0) {
        printf("%ld\n", i0);
    }
    /* potentially more work */
}
```

The complete example code is available in [print-numbers-openacc.cpp](../src/print-numbers/print-numbers-openacc.cpp).
Build and execute them using the following cells.

In [ ]:
!nvc++ -O3 -std=c++17 -acc=gpu -target=gpu -o ../build/print-numbers/print-numbers-openacc ../src/print-numbers/print-numbers-openacc.cpp

In [ ]:
!../build/print-numbers/print-numbers-openacc

## Level 1: Adding Managed Memory

Our next application is increasing all elements of an array by one.

[increase-base.cpp](../src/increase/increase-base.cpp) shows a serial CPU-only implementation.
Its key part and our entry point is the increase function.

```c++
void increase(double* data, size_t nx) {
    for (size_t i0 = 0; i0 < nx; ++i0) {
        data[i0] += 1;
    }
}
```

As with OpenMP target offloading, the parallelization approach follows the previously discussed concepts.
Support for managed memory is enabled by adding the `-gpu=mem:managed` compiler switch.

The complete example code can be found in [increase-openacc-mm.cpp](../src/increase/increase-openacc-mm.cpp), and be built and executed with the following cells.

In [ ]:
!nvc++ -O3 -std=c++17 -acc=gpu -target=gpu -gpu=mem:managed -o ../build/increase/increase-openacc-mm ../src/increase/increase-openacc-mm.cpp

In [ ]:
!../build/increase/increase-openacc-mm

## Level 2: Switching to Explicit Memory Management

OpenACC follows the same principles as OpenMP for explicit data management, but does so with a slightly different syntax.

```c++
#pragma acc enter data copyin (field[0:nx])
#pragma acc exit  data copyout(field[0:nx])
```

The complete example code is available in [increase-openacc-expl.cpp](../src/increase/increase-openacc-expl.cpp).
Build and execute it using the following cells.

In [ ]:
!nvc++ -O3 -std=c++17 -acc=gpu -target=gpu -o ../build/increase/increase-openacc-expl ../src/increase/increase-openacc-expl.cpp

In [ ]:
!../build/increase/increase-openacc-expl

## Exercise: Initialization on the GPU

So far, the data is initialized on the host by the function `init` and then transferred to the GPU.
Adapt the explicit memory version from level 2 such that the data is initialized directly on the GPU instead:

* Port `init` to the GPU, analogous to `increase`.
* Replace the host-to-device transfer by a device allocation only.

Keep the transfer back to the host since the result is still checked there.
If no error is reported when running the application, your implementation is correct.

Besides the time for the measured iterations (`elapsed time`), the application reports its end-to-end runtime (`total time`) and the overhead, i.e. everything except the measured iterations, such as runtime start-up, allocation, initialization, data transfers, warm-up, the check and deallocation.
Compare both with the original version above.

**Hints**

* `create` allocates device memory without transferring any data.
* `present` asserts that data is already on the device.
* The device allocation now has to happen before `init` is called.

Copy the example into the working file [ex-increase-openacc-expl.cpp](../src/increase/ex-increase-openacc-expl.cpp) with the cell below, then adapt it there.

In [ ]:
%%bash
if [ -e ../src/increase/ex-increase-openacc-expl.cpp ]; then
  echo "error: target file already exists"
else
  cp ../src/increase/increase-openacc-expl.cpp ../src/increase/ex-increase-openacc-expl.cpp
fi

### Possible Solution

[sol-increase-openacc-expl.cpp](../src/increase/sol-increase-openacc-expl.cpp) contains a possible solution for this exercise.
Copy it into the working file [ex-increase-openacc-expl.cpp](../src/increase/ex-increase-openacc-expl.cpp) with the cell below.

In [ ]:
%%bash
if [ -e ../src/increase/ex-increase-openacc-expl.cpp ]; then
  echo "error: target file already exists"
else
  cp ../src/increase/sol-increase-openacc-expl.cpp ../src/increase/ex-increase-openacc-expl.cpp
fi

### Compilation and Execution

Build and execute the working file [ex-increase-openacc-expl.cpp](../src/increase/ex-increase-openacc-expl.cpp) using the following cells.

In [ ]:
!nvc++ -O3 -std=c++17 -acc=gpu -target=gpu -Minfo=accel -o ../build/increase/ex-increase-openacc-expl ../src/increase/ex-increase-openacc-expl.cpp

In [ ]:
!../build/increase/ex-increase-openacc-expl